# Обработка входных данных

In [3]:
import pandas as pd; import numpy as np
df1=pd.read_csv('raw_dataset.txt')
# Очистка датасета
## Очистка основных данных
#Добавляем признак - находится ли пловец в родных стенах
df1['is_swimmer_in_home_country']=((df1['swimmer_country_code'] == df1['host_country_code']) & (~df1['swimmer_country_code'].isna())).astype('int')
df1['host_city'].value_counts()
#Удаляем строки где target переменная result_time=nan
df1=df1.dropna(subset=['result_time'])

#Переводим результирующее время в секунды
def get_time_in_seconds(x):
    if ':' in x: #Формат ММ:СС.МС или ЧЧ:ММ:СС.МС

        split=x.split(':')
        if len(split)>2:
            return float(split[0])*60*60+float(split[1])*60+float(split[2])

        return float(split[0])*60+float(split[1])
    return float(x)

# 4. Считаем итоговое время в секундах
df1['result_time'] = df1['result_time'].apply(get_time_in_seconds)
## Удалить строки, где result_time и heat_rank несогласованы друг с другом (пловец находится между двумя пловцами по рангу, но по времени он не находится между ними)
#Проверить пустые значения
df1.isna().sum()
#Удалить строки, где пловец x находится между двумя пловцами a и b по рангу, rank(a)<rank(x)<rank(b), но не выполняется логичное условие времени: time(a)<time(x)<time(b)
#Сначала найдем эти строки
race_results_cache={}
def find_inconsistent_rows(row:pd.Series):
    global df1,race_results_cache #Весь датафрейм
    #Если какая-то необходимая колонка отсутствует - помечаем столбец как консистентный.
    if pd.isna(row['heat_rank']) or pd.isna(row['result_time']) or pd.isna(row['competition_name']) \
        or pd.isna(row['discipline_name']) or pd.isna(row['race_number_in_phase']) or pd.isna(row['phase_name']) or pd.isna(row['race_date_local']):
            return 0
    h_r=row['heat_rank']
    r_t=row['result_time']
    c_n=row['competition_name']
    d_n=row['discipline_name']
    h_n=row['race_number_in_phase']
    p_n=row['phase_name']
    r_d=row['race_date_local']
    
    race_results=None
    key=(c_n,d_n,h_n,p_n,r_d)
    if key in race_results_cache:
        race_results=race_results_cache[key]
    else:
        #Строки пловцов из данного заплыва отсортированные по месту пловца в заплыве
        race_results=df1[(df1['competition_name']==c_n) & (df1['discipline_name']==d_n) & (df1['race_date_local']==r_d) &
            (df1['race_number_in_phase']==h_n) & (df1['phase_name']==p_n) & (~pd.isna(df1['heat_rank'])) & (~pd.isna(df1['result_time']))].sort_values('heat_rank')
        race_results_cache[key]=race_results
    
    #Индекс худшего по месту пловца
    right_idx=race_results['heat_rank'].searchsorted(h_r,side='right')
    
    #Индекс лучшего по месту пловца
    left_idx=race_results['heat_rank'].searchsorted(h_r,side='left') -1
    
    #Индексы вышли за границы - пловец не находится между двумя другими в результатах заплыва. Считаем, что консистентно
    if left_idx<0 or right_idx>=len(race_results):
        return 0
    
    right_row=race_results.iloc[right_idx]
    left_row=race_results.iloc[left_idx]
    #Если время лучшего пловца относительно текущего больше или равно текущему пловцу
    # или время текущего пловца больше или равно времени худшего пловца - в данных неконсистентность. Удаляем эту строку.
    if ((left_row['result_time']>=row['result_time']) | (row['result_time']>=right_row['result_time'])):
        return 1
    #Иначе возвращаем 0
    return 0

df1['is_heat_rank_result_time_inconsistent']=df1.apply(find_inconsistent_rows,axis=1)
df1[df1['is_heat_rank_result_time_inconsistent']==1]
df1['is_heat_rank_result_time_inconsistent'].value_counts()
#Удалить строки, где результирующее время неконсистентно с итоговым местом пловца
df1=df1[df1['is_heat_rank_result_time_inconsistent']==0]

#Удаляем столбец is_heat_rank_result_time_inconsistent, так как он больше не нужен. Также удаляем столбец heat_rank, так как он содержит данные, которые недоступны на момент начала соревнования
df1=df1.drop(columns=['is_heat_rank_result_time_inconsistent','heat_rank'])
## Продолжение очистки
#Делим признак discipline_name на 3 признака: пол пловца, дистанция и стиль плавания
splitted_discipline_name=df1['discipline_name'].str.split(expand=True)
df1['swimmer_sex']=splitted_discipline_name[0].map({"Men's":0,"Women's":1})
df1["distance"]=splitted_discipline_name[1].str.replace('m','')
df1['distance']=pd.to_numeric(df1['distance'])
df1["style"]=splitted_discipline_name[2]
df1=df1.drop(columns="discipline_name")


#Удаляем все строки, где или пол, или дистанция или стиль плавания неизвестен
df1=df1.dropna(subset=['swimmer_sex','style','distance'])



#Делаем OHE для стиля плавания
df1=pd.concat([df1,pd.get_dummies(df1['style'],dtype=int)],axis=1)
df1=df1.drop(columns='style')

#Удаляем host_city, т.к. не придумал, что делать с ним при использовании модели (если попадётся неизвестный город).
# + Городов много, и врядли какие-то важные зависимости найдутся на конкретные города (для 1 города мало записей)
df1=df1.drop(columns=['host_city'])

#Удаляем из датасета имя пловца и имя соревнования (имя соревнования не поможет предсказать будущие соревнования. Имя пловца дублируется его id)
df1=df1.drop(columns=['competition_name','swimmer_full_name'])
#Делаем OHE для региона страны проведения
df1=pd.concat([df1,pd.get_dummies(df1['host_region'],dtype=int)],axis=1)
df1=df1.drop(columns='host_region')
df1['phase_name'].value_counts()
df1=df1[~df1['phase_name'].isin(['Swim Off Heats', 'Swim Off Semifinals'])] #Удалим записи связанные с дополнительными заплывами за выход в финал или полуфинал, т.к. их мало
#Делаем OHE для фазы проведения
df1=pd.concat([df1,pd.get_dummies(df1['phase_name'],dtype=int)],axis=1)
df1=df1.drop(columns='phase_name')
#Переводим длину бассейна в целое число
df1['pool_length']=df1['pool_configuration'].str.replace('m','').astype(int)
df1=df1.drop(columns='pool_configuration')
df1=df1[(df1['swimmer_lane'].isna()) | ((df1['swimmer_lane']>=0) & (df1['swimmer_lane']<=10))] #Удалим записи где № дорожки пловца <0 или >10
#Удаляем вес, так как там слишком много пропусков
df1=df1.drop(columns='swimmer_weight')
#Заполняем пустые значения строкой "Unknown"
df1['swimmer_id']=df1['swimmer_id'].fillna("Unknown").astype(str)
df1['host_country_code']=df1['host_country_code'].fillna("Unknown").astype(str)
df1['swimmer_country_code']=df1['swimmer_country_code'].fillna("Unknown").astype(str)
## Кодируем временные признаки через sin cos encoding и обрабатываем возраст пловцов
#Удаляем строки, где возраст пловца <12 или >60 (они скорее всего "мусорные"). Пропуски оставляем
df1=df1[(df1['swimmer_age_at_swim_start'].isna()) |((~df1['swimmer_age_at_swim_start'].isna()) & (df1['swimmer_age_at_swim_start']>=12) & (df1['swimmer_age_at_swim_start']<=60))]

#Заполняем возраст в строках, где возраст пловца неизвестен, медианой возрастов
median=df1['swimmer_age_at_swim_start'].median()
print('age median:',median)
df1['swimmer_age_at_swim_start']=df1['swimmer_age_at_swim_start'].fillna(median)
df1
#Добавляем переменные has_race_date_local - для отображения, были ли в датасете точные даты заплыва
df1['has_race_date_local']=df1['race_date_local'].notna().astype(int)


#Где есть дата заплыва, получаем из даты год; синус месяца в году; косинус месяца в году; синус дня года; косинус дня года; синус дня в неделе; косинус дня в неделе;
#Где нет - заполняем медианами имеющихся значений

dates=pd.to_datetime(df1['race_date_local'],errors='coerce')
year=dates.dt.year
month=dates.dt.month
doy=dates.dt.day_of_year
dow=dates.dt.day_of_week
days_in_year=365+dates.dt.is_leap_year.astype(int)
date_columns=pd.DataFrame({
    'race_year': year,
    'race_month_sin': np.sin(2 * np.pi * month / 12),
    'race_month_cos': np.cos(2 * np.pi * month / 12),
    'race_doy_sin'  : np.sin(2 * np.pi * doy / days_in_year),
    'race_doy_cos'  : np.cos(2 * np.pi * doy / days_in_year),
    'race_dow_sin'  : np.sin(2 * np.pi * dow / 7),
    'race_dow_cos'  : np.cos(2 * np.pi * dow / 7)
})
for col in date_columns:
    median=date_columns[col].median()
    date_columns[col]=date_columns[col].fillna(median).astype(float)
df1=pd.concat([df1.drop(columns='race_date_local'),date_columns],axis=1) #также удаляем исходную колонку с датой заплыва
df1

#Добавляем переменные has_race_time_local - для отображения, были ли в датасете точные времена заплыва
df1['has_race_time_local']=df1['race_time_local'].notna().astype(int)
#Где есть время заплыва, получаем из времени синус секунд дня; косинус секунд дня;
#Где нет - заполняем синусом и косинусам секунд в 12:00:00
times=pd.to_timedelta(df1['race_time_local'],errors='coerce')
seconds=times.dt.total_seconds()
seconds_in_day=60*60*24
time_columns=pd.DataFrame({
    'race_time_seconds_sin': np.sin(seconds*2*np.pi/seconds_in_day),
    'race_time_seconds_cos': np.cos(seconds*2*np.pi/seconds_in_day),
})
sin_secs_in_12_00_00=np.sin(12*60*60*2*np.pi/seconds_in_day)
cos_secs_in_12_00_00=np.cos(12*60*60*2*np.pi/seconds_in_day)
time_columns['race_time_seconds_sin']=time_columns['race_time_seconds_sin'].fillna(sin_secs_in_12_00_00).astype(float)
time_columns['race_time_seconds_cos']=time_columns['race_time_seconds_cos'].fillna(cos_secs_in_12_00_00).astype(float)
df1=pd.concat([df1.drop(columns='race_time_local'),time_columns],axis=1) #также удаляем исходную колонку с временем
df1
#Добавляем переменные has_swimmer_dob - для отображения, были ли в датасете даты рождения пловца
df1['has_swimmer_dob']=df1['swimmer_date_of_birth'].notna().astype(int)


#Где есть дата рождения пловца, получаем из даты год; синус месяца в году; косинус месяца в году; синус дня года; косинус дня года; синус дня в неделе; косинус дня в неделе;
#Где нет - заполняем медианами имеющихся значений

dates=pd.to_datetime(df1['swimmer_date_of_birth'],errors='coerce')
year=dates.dt.year
month=dates.dt.month
doy=dates.dt.day_of_year
dow=dates.dt.day_of_week
days_in_year=365+dates.dt.is_leap_year.astype(int)
date_columns=pd.DataFrame({
    'swimmer_dob_year': year,
    'swimmer_dob_month_sin': np.sin(2 * np.pi * month / 12),
    'swimmer_dob_month_cos': np.cos(2 * np.pi * month / 12),
    'swimmer_dob_doy_sin'  : np.sin(2 * np.pi * doy / days_in_year),
    'swimmer_dob_doy_cos'  : np.cos(2 * np.pi * doy / days_in_year),
    'swimmer_dob_dow_sin'  : np.sin(2 * np.pi * dow / 7),
    'swimmer_dob_dow_cos'  : np.cos(2 * np.pi * dow / 7)
})
for col in date_columns:
    median=date_columns[col].median()
    date_columns[col]=date_columns[col].fillna(median).astype(float)
df1=pd.concat([df1.drop(columns='swimmer_date_of_birth'),date_columns],axis=1) #также удаляем исходную колонку с датой рождения пловца
df1


## Обработка столбцов swimmer_lane и swimmer_height
#Добавляем переменные has_swimmer_lane - для отображения, были ли в датасете номера дорожки пловца
df1['has_swimmer_lane']=df1['swimmer_lane'].notna().astype(int)

#Заполняем отсутствующие значения в столбце swimmer_lane медианами
median=df1['swimmer_lane'].median()

df1['swimmer_lane']=df1['swimmer_lane'].fillna(median).astype(float)
#Добавляем переменные has_swimmer_height - для отображения, были ли в датасете росты пловцов
df1['has_swimmer_height']=df1['swimmer_height'].notna().astype(int)

#Заполняем отсутствующие значения в столбце swimmer_height медианами
median=df1['swimmer_height'].median()
print('swimmer_height median:',median)
df1['swimmer_height']=df1['swimmer_height'].fillna(median).astype(float)

age median: 20.0
swimmer_height median: 180.0


# Создание эмбеддингов

In [4]:
#Делим на X и Y
X=df1.drop(columns='result_time')
Y=df1['result_time']

In [5]:
from tensorflow.keras.layers import Embedding, Input, Dense, Flatten, Concatenate, LeakyReLU, StringLookup
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
#Описываем входы
swimmer_country_input = Input(shape=(1,),dtype='string', name='swimmer_country')
host_country_input = Input(shape=(1,),dtype='string', name='host_country')
swimmer_id_input = Input(shape=(1,),dtype='string', name='swimmer_id')
other_featues_input = Input(shape=(len(df1.columns)-4,), name='other_features') #убираем таргет, id и 2 country

#Переводы словарями из строки в числа на вход к эмбеддингам
swimmer_country_lookup = StringLookup(vocabulary=np.setdiff1d(df1['swimmer_country_code'].unique(),['Unknown']),output_mode='int')
host_country_lookup = StringLookup(vocabulary=np.setdiff1d(df1['host_country_code'].unique(),['Unknown']),output_mode='int')
swimmer_id_lookup = StringLookup(vocabulary=np.setdiff1d(df1['swimmer_id'].unique(),['Unknown']),output_mode='int')

#Описываем эмбеддинги
swimmer_country_index = swimmer_country_lookup(swimmer_country_input)
swimmer_country_emb = Embedding(swimmer_country_lookup.vocabulary_size(), 8,name ="swimmer_country_emb")(swimmer_country_index)
swimmer_country_emb = Flatten()(swimmer_country_emb)

host_country_index = host_country_lookup(host_country_input)
host_country_emb = Embedding(host_country_lookup.vocabulary_size(), 8,name ="host_country_emb")(host_country_index)
host_country_emb = Flatten()(host_country_emb)

swimmer_id_index = swimmer_id_lookup(swimmer_id_input)
swimmer_id_emb = Embedding(swimmer_id_lookup.vocabulary_size(), 16,name ="swimmer_id_emb")(swimmer_id_index)
swimmer_id_emb = Flatten()(swimmer_id_emb)



#Объединяем все слои
x = Concatenate()([swimmer_country_emb, host_country_emb, swimmer_id_emb, other_featues_input])
x = Dense(128, activation=LeakyReLU(alpha=0.01))(x)
x = Dense(1)(x)  #Выходной слой

model = Model(inputs=[swimmer_country_input, host_country_input, swimmer_id_input, other_featues_input], outputs=x)



c:\Users\den\Desktop\vkr_train_models\.venv\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [6]:
#Обучаем модель
model.compile(
    optimizer='adam',
    loss='mae',
    metrics=['mae','mape']
)
import keras
import tensorflow as tf
tf.debugging.set_log_device_placement(True)
model.fit(
    x={
        'swimmer_country': tf.constant(X['swimmer_country_code'].values, dtype=tf.string),
        'host_country': tf.constant(X['host_country_code'].values, dtype=tf.string),
        'swimmer_id': tf.constant(X['swimmer_id'].values, dtype=tf.string),
        'other_features': X.drop(columns=['swimmer_country_code','host_country_code','swimmer_id']).astype(np.float32).values
    },
    y=Y.values,
    validation_data=(
        {
            'swimmer_country': tf.constant(X['swimmer_country_code'].values, dtype=tf.string),
            'host_country': tf.constant(X['host_country_code'].values, dtype=tf.string),
            'swimmer_id': tf.constant(X['swimmer_id'].values, dtype=tf.string),
            'other_features': X.drop(columns=['swimmer_country_code','host_country_code','swimmer_id']).astype(np.float32).values
        },
        Y.values
    ),
    epochs=100,
    batch_size=128,
)

Epoch 1/100
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 30s 12ms/step - loss: 6.9738 - mae: 6.9738 - mape: 7.9044 - val_loss: 6.8851 - val_mae: 6.8851 - val_mape: 9.4173
Epoch 2/100
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - loss: 4.8093 - mae: 4.8093 - mape: 5.6580 - val_loss: 4.6643 - val_mae: 4.6643 - val_mape: 6.2031
Epoch 3/100
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - loss: 4.1388 - mae: 4.1388 - mape: 4.9946 - val_loss: 4.6156 - val_mae: 4.6156 - val_mape: 6.3916
Epoch 4/100
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - loss: 3.8249 - mae: 3.8249 - mape: 4.7250 - val_loss: 3.5475 - val_mae: 3.5475 - val_mape: 4.1194
Epoch 5/100
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - loss: 3.5285 - mae: 3.5285 - mape: 4.4902 - val_loss: 5.2349 - val_mae: 5.2349 - val_mape: 8.1192
Epoch 6/100
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - loss: 3.3964 - mae: 3.3964 - mape: 4.4215 - val_loss: 3.2900 - val_mae: 3.2900 - val_mape: 4.6443
Epoch 7/100
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step -

In [7]:
#Получить веса из эмбеддингов
swimmer_country_emb_weights=model.get_layer('swimmer_country_emb').get_weights()[0]
host_country_emb_weights=model.get_layer('host_country_emb').get_weights()[0]
swimmer_id_emb_weights=model.get_layer('swimmer_id_emb').get_weights()[0]

#Получить словари категория → вектор
swimmer_country_vocab = swimmer_country_lookup.get_vocabulary()
host_country_vocab = host_country_lookup.get_vocabulary()
swimmer_id_vocab = swimmer_id_lookup.get_vocabulary()

#Получить словари категория -> вектор обученного эмбеддинга
swimmer_country_to_vector = {cat: swimmer_country_emb_weights[i] for i, cat in enumerate(swimmer_country_vocab)}
host_country_to_vector = {cat: host_country_emb_weights[i] for i, cat in enumerate(host_country_vocab)}
swimmer_id_to_vector = {cat: swimmer_id_emb_weights[i] for i, cat in enumerate(swimmer_id_vocab)}

In [8]:
df2=df1
#Добавляем обученные эмбеддинги страны пловца в датасет


def map_to_emb_df(column, vector_dict,col_prefix): #Получить из столбца dataframe, каждая строка соответствует эмбеддингу
    default = vector_dict['[UNK]']
    matrix=np.stack(column.map(lambda x: vector_dict.get(x, default)).values)
    return pd.DataFrame(matrix,columns=[f'{col_prefix}_{i}' for i in range(matrix.shape[1])],index=column.index)


#Получить датафреймы эмбеддингов для строк датасета (число строк датафрейма = число строк датасета, число столбцов = длина эмбеддинга)
swimmer_country_df = map_to_emb_df(df2['swimmer_country_code'], swimmer_country_to_vector,'swimmer_country')
host_country_df = map_to_emb_df(df2['host_country_code'], host_country_to_vector,'host_country')
swimmer_id_df = map_to_emb_df(df2['swimmer_id'], swimmer_id_to_vector,'swimmer_id')


#Добавить полученные эмбеддинги в датасет
df2 = pd.concat([df2.drop(columns=['swimmer_country_code', 'host_country_code']),
                 swimmer_country_df,host_country_df,swimmer_id_df], axis=1)

# Обучение модели BiLSTM

In [9]:
#Снова разделить на X и Y
X_BiLSTM=df2.drop(columns='result_time')
Y_BiLSTM=df2['result_time']

In [10]:
#Посмотреть порядок колонок
print(list(X.columns))

['host_country_code', 'race_number_in_phase', 'races_in_phase', 'swimmer_country_code', 'swimmer_age_at_swim_start', 'swimmer_id', 'swimmer_lane', 'swimmer_height', 'is_swimmer_in_home_country', 'swimmer_sex', 'distance', 'Backstroke', 'Breaststroke', 'Butterfly', 'Freestyle', 'Medley', 'Africa', 'Americas', 'Asia', 'Europe', 'Oceania', 'Finals', 'Heats', 'Semifinals', 'pool_length', 'has_race_date_local', 'race_year', 'race_month_sin', 'race_month_cos', 'race_doy_sin', 'race_doy_cos', 'race_dow_sin', 'race_dow_cos', 'has_race_time_local', 'race_time_seconds_sin', 'race_time_seconds_cos', 'has_swimmer_dob', 'swimmer_dob_year', 'swimmer_dob_month_sin', 'swimmer_dob_month_cos', 'swimmer_dob_doy_sin', 'swimmer_dob_doy_cos', 'swimmer_dob_dow_sin', 'swimmer_dob_dow_cos', 'has_swimmer_lane', 'has_swimmer_height']


In [11]:
#Посмотреть порядок колонок
print(list(X_BiLSTM.columns))

['race_number_in_phase', 'races_in_phase', 'swimmer_age_at_swim_start', 'swimmer_id', 'swimmer_lane', 'swimmer_height', 'is_swimmer_in_home_country', 'swimmer_sex', 'distance', 'Backstroke', 'Breaststroke', 'Butterfly', 'Freestyle', 'Medley', 'Africa', 'Americas', 'Asia', 'Europe', 'Oceania', 'Finals', 'Heats', 'Semifinals', 'pool_length', 'has_race_date_local', 'race_year', 'race_month_sin', 'race_month_cos', 'race_doy_sin', 'race_doy_cos', 'race_dow_sin', 'race_dow_cos', 'has_race_time_local', 'race_time_seconds_sin', 'race_time_seconds_cos', 'has_swimmer_dob', 'swimmer_dob_year', 'swimmer_dob_month_sin', 'swimmer_dob_month_cos', 'swimmer_dob_doy_sin', 'swimmer_dob_doy_cos', 'swimmer_dob_dow_sin', 'swimmer_dob_dow_cos', 'has_swimmer_lane', 'has_swimmer_height', 'swimmer_country_0', 'swimmer_country_1', 'swimmer_country_2', 'swimmer_country_3', 'swimmer_country_4', 'swimmer_country_5', 'swimmer_country_6', 'swimmer_country_7', 'host_country_0', 'host_country_1', 'host_country_2', 'hos

In [ ]:
from tensorflow.keras.regularizers import l1_l2
from keras import layers
from keras import activations
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from functools import partial
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
from tensorflow.keras.metrics import R2Score
from random import seed
from os import environ
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import TransformedTargetRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Masking,Bidirectional,LSTM,Dense,Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from numpy.lib.stride_tricks import sliding_window_view
import tensorflow as tf
import numpy as np
import joblib



#Класс для нормализации входных признаков и переделки их в формат, подходящий на вход к BiLSTM
class SequenceStandardScalerForBiLSTM(BaseEstimator, TransformerMixin):
    def __init__(self,
                 window_size=5,                           #Размер окна для последовательности
                 ):
        self.window_size = window_size                
        self.is_train_data_transformed=False             #Флаг, показывает, были ли преобразованы тренировочные данные через transform
        
    #Вернуть только те колонки датафрейма, которые не являются строками или объектами
    def _get_required_columns_from_dataset(self, X):
        return X.select_dtypes(include='number').columns.tolist()

    def fit(self, X, y=None):
        #Обучаем StandardScaler для признаков на тренировочном датасете
        self.feature_cols = self._get_required_columns_from_dataset(X)
        self.scaler_ = StandardScaler().fit(X[self.feature_cols])     
        self.is_train_data_transformed = False

        return self

    def transform(self, X):
        if not self.is_train_data_transformed:  #На train данных
            X_scaled = X.copy()
        
            #Получить углы дня года и времени в секундах дня - для сортировки записей в хронологическом порядке
            X_scaled['_doy_angle'] = np.arctan2(X['race_doy_sin'], X['race_doy_cos']) % (2 * np.pi)
            X_scaled['_time_angle'] = np.arctan2(X['race_time_seconds_sin'], X['race_time_seconds_cos']) % (2 * np.pi)
            
            #Преобразуем значения внутри датафрейма согласно StandardScaler
            X_scaled[self.feature_cols] = self.scaler_.transform(X[self.feature_cols])
            
            X_scaled['_original_idx'] = np.arange(len(X))  
            
            #Сортируем датафрейм сначала по ИД пловца, затем по году заплыва, затем по углу дня года, затем по углу времени дня
            X_scaled = X_scaled.sort_values(by=['swimmer_id', 'race_year', '_doy_angle', '_time_angle'])
            
            #Есть проблема, что не у всех строк в датасете было время (оно было импутировано для некоторых значений).
            #Однако дата есть у всех строк. В целом же погрешность времени в течении дня не должно сильно влиять на производительность модели
            
            
            #Выходной 3D массив на вход к модели
            sequences = np.zeros((len(X), self.window_size, len(self.feature_cols)), dtype=np.float32)
            
            #Получаем последовательности для обучения BiLSTM и сохраняем 
            self.swimmer_history = {}            
            for swimmer_id, group in X_scaled.groupby('swimmer_id', sort=False):
                X_for_1_swimmer = group[self.feature_cols].values.astype(np.float32)                      #Отсортированный массив результатов одного пловца
                paddings=np.full((self.window_size-1, len(self.feature_cols)), -999.0, dtype=np.float32)  #(windows_size-1,n_features)
                swimmer_history_sequence=np.vstack([paddings,X_for_1_swimmer])                             #(len(X_for_1_swimmer)+windows_size-1,n_features) 
                original_indices = group['_original_idx'].values                              
                
                windows = sliding_window_view(swimmer_history_sequence, window_shape=self.window_size, axis=0) #(len(X_for_1_swimmer),n_features,windows_size)
                
                #Переставить местами два измерения массива, так как BiLSTM ожидает массив размерности (batch,timesteps,n_features)
                windows = np.swapaxes(windows, 1, 2) #(len(X_for_1_swimmer),windows_size,n_features)
                
                #Присваиваем результирующие последовательности для обучения BiLSTM внутрь выходного 3Д массива
                sequences[original_indices] = windows 
                
                self.swimmer_history[swimmer_id] = X_for_1_swimmer[-(self.window_size - 1):]      #Сохраняем последние записи пловца в историю
                
            self.is_train_data_transformed=True
            return sequences
        else:   #На test/validation данных
            X_scaled = X.copy()
        
            #Получить углы дня года и времени в секундах дня - для сортировки записей в хронологическом порядке
            X_scaled['_doy_angle'] = np.arctan2(X['race_doy_sin'], X['race_doy_cos']) % (2 * np.pi)
            X_scaled['_time_angle'] = np.arctan2(X['race_time_seconds_sin'], X['race_time_seconds_cos']) % (2 * np.pi)
            
            #Преобразуем значения внутри датафрейма согласно StandardScaler
            X_scaled[self.feature_cols] = self.scaler_.transform(X[self.feature_cols])
            
            X_scaled['_original_idx'] = np.arange(len(X)) 
            
            #Сортируем датафрейм сначала по ИД пловца, затем по году заплыва, затем по углу дня года, затем по углу времени дня
            X_scaled = X_scaled.sort_values(by=['swimmer_id', 'race_year', '_doy_angle', '_time_angle'])
                                   
            #Выходной 3D массив на вход к модели
            sequences = np.zeros((len(X), self.window_size, len(self.feature_cols)), dtype=np.float32)
            
            #Получаем последовательности для обучения BiLSTM и сохраняем             
            for swimmer_id, group in X_scaled.groupby('swimmer_id', sort=False):
                X_for_1_swimmer = group[self.feature_cols].values.astype(np.float32)                      #Отсортированный массив результатов одного пловца
                
                #Пытаемся получить историю пловца
                history = self.swimmer_history.get(
                    swimmer_id,
                    np.zeros((0, len(self.feature_cols)), dtype=np.float32)
                )
                
                padding_size=max(0,self.window_size-1-len(history))  #Размер паддинга слева значениями -999
                
                #Строим массив последовательностей
                swimmer_sequences=None
                if padding_size>0:
                    paddings=np.full((padding_size, len(self.feature_cols)), -999.0, dtype=np.float32)   #(padding_size,n_features)
                    swimmer_sequences=np.vstack([paddings,history,X_for_1_swimmer])                     #(len(X_for_1_swimmer)+windows_size-1,n_features)
                else:
                    swimmer_sequences=np.vstack([history,X_for_1_swimmer])                     #(len(X_for_1_swimmer)+windows_size-1,n_features)
                original_indices = group['_original_idx'].values                              
                
                windows = sliding_window_view(swimmer_sequences, window_shape=self.window_size, axis=0) #(len(X_for_1_swimmer),n_features,windows_size)
                
                #Переставить местами два измерения массива, так как BiLSTM ожидает массив размерности (batch,timesteps,n_features)
                windows = np.swapaxes(windows, 1, 2) #(len(X_for_1_swimmer),windows_size,n_features)
                
                #Присваиваем результирующие последовательности для обучения BiLSTM внутрь выходного 3Д массива
                sequences[original_indices] = windows               
                
            return sequences

In [ ]:
SEED=11
#Описание модели
def build_model_ablation_original_BiLSTM(input_shape,
                                         num_bilstm_layers=1,
                                   lstm_units_in_layer=16,
                                   dropout_rate=0,
                                   learning_rate=0.001,
                                   loss_function='mse'):

    tf.random.set_seed(SEED)

    model = Sequential()
    tf.random.set_seed(SEED);model.add(Input(shape=input_shape))
    
    #Значения, маскированные -999 (пустые значения, которыми слева заполнялись последовательности внутри BiLSTM) будут игнорироваться при обучении
    tf.random.set_seed(SEED);model.add(Masking(mask_value=-999.0))
    
    

    for i in range(num_bilstm_layers):
        if i<num_bilstm_layers-1:
            tf.random.set_seed(SEED);model.add(Bidirectional(LSTM(lstm_units_in_layer, return_sequences=True)))  #На выходе - 3d тензор. Передаем его для следующего BiLSTM слоя
        else:
            tf.random.set_seed(SEED);model.add(Bidirectional(LSTM(lstm_units_in_layer, return_sequences=False))) #На выходе - 2d тензор. Передаем его для полносвязного слоя
        if dropout_rate > 0:
            tf.random.set_seed(SEED);model.add(Dropout(dropout_rate))


    tf.random.set_seed(SEED);model.add(Dense(1, activation='linear'))

    model.compile(optimizer=Adam(learning_rate=learning_rate),loss=loss_function, metrics=['mean_absolute_error', 'mean_absolute_percentage_error', 'r2_score'])
    return model

In [ ]:
#Фиксируем случайность
seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()
environ['TF_DETERMINISTIC_OPS'] = '1'

# Делаем все без пайплайна
features_preprocessor=SequenceStandardScalerForBiLSTM(window_size=7)
target_transformer=StandardScaler()


#Фит
X_preprocessed=features_preprocessor.fit_transform(X_BiLSTM)
Y_transformed=target_transformer.fit_transform(Y_BiLSTM.values.reshape(-1, 1)).ravel()
BiLSTM_model=build_model_ablation_original_BiLSTM(input_shape=X_preprocessed.shape[1:],
                dropout_rate=0.1,
                learning_rate=0.001,
                loss_function='mae',
                lstm_units_in_layer=128,
                num_bilstm_layers=2,)
BiLSTM_model.fit(X_preprocessed,Y_transformed,
                batch_size=128,
                epochs=25,
                verbose=1)


Epoch 1/25
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 61s 22ms/step - loss: 0.0657 - mean_absolute_error: 0.0657 - mean_absolute_percentage_error: 74.5061 - r2_score: 0.9686
Epoch 2/25
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 57s 23ms/step - loss: 0.0300 - mean_absolute_error: 0.0300 - mean_absolute_percentage_error: 28.7969 - r2_score: 0.9964
Epoch 3/25
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 58s 23ms/step - loss: 0.0272 - mean_absolute_error: 0.0272 - mean_absolute_percentage_error: 25.5840 - r2_score: 0.9969
Epoch 4/25
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 56s 23ms/step - loss: 0.0255 - mean_absolute_error: 0.0255 - mean_absolute_percentage_error: 25.0573 - r2_score: 0.9973
Epoch 5/25
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 56s 23ms/step - loss: 0.0239 - mean_absolute_error: 0.0239 - mean_absolute_percentage_error: 24.2697 - r2_score: 0.9975
Epoch 6/25
2470/2470 ━━━━━━━━━━━━━━━━━━━━ 56s 23ms/step - loss: 0.0220 - mean_absolute_error: 0.0220 - mean_absolute_percentage_error: 22.9651 - r2_score: 0.9977
Epoch 7/25
2470/2470 ━━━━━━━

In [65]:
Y_pred=target_transformer.inverse_transform(BiLSTM_model.predict(X_preprocessed))

9879/9879 ━━━━━━━━━━━━━━━━━━━━ 44s 4ms/step


In [66]:
#Проверка что модель нормально обучилась
print("MAE на трейне у итоговой модели:", mean_absolute_error(Y_BiLSTM, Y_pred), sep="")
print("MAPE на трейне у итоговой модели:", mean_absolute_percentage_error(Y_BiLSTM, Y_pred) * 100, "%", sep="")
print("R^2 на трейне у итоговой модели:", r2_score(Y_BiLSTM, Y_pred), sep="")

MAE на трейне у итоговой модели:1.472737728537871
MAPE на трейне у итоговой модели:1.378048453865259%
R^2 на трейне у итоговой модели:0.9995629460096151


In [ ]:
import joblib
#Сохранить словари эмбеддингов
joblib.dump(swimmer_country_to_vector,"swimmer_country_embedds.joblib")
joblib.dump(host_country_to_vector,"host_country_embedds.joblib")
joblib.dump(swimmer_id_to_vector,"swimmer_id_embedds.joblib")

['swimmer_id_embedds.joblib']

In [79]:

#Сохраним нейросеть
BiLSTM_model.save('BiLSTM_model.keras')

#Сохранить препроцессор
joblib.dump(features_preprocessor,'features_preprocessor.joblib')

#Сохранить преобразователь таргета
joblib.dump(target_transformer,'target_transformer.joblib')

['target_transformer.joblib']

In [ ]:
#Также скопировал вручную код функции SequenceStandardScalerForBiLSTM в файл SequenceStandardScalerForBiLSTM.py